# Gold layer Transformation Logic

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
query = """
SELECT 
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) as customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    case
        when ci.gender <> 'n/a' then ci.gender
        else coalesce(ca.gender, 'n/a')
    end as gender,
    ca.birth_date as birthdate,
    ci.created_date as create_date
FROM silver.crm_customers ci
left join silver.erp_customers ca
    on ci.customer_number = ca.customer_number
left join silver.erp_customer_location la
    on ci.customer_number = la.customer_number
"""

df = spark.sql(query)


In [0]:
df.limit(10).display()

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

# Sanity check of gold table

In [0]:
%sql
select * from workspace.gold.dim_customers limit 10